**The objective of this Notebook is to create an environment to simulate various **Load Balancing parameters** i.e. throughput, performance, migration time,
response time, overhead, resource usage, scalability, fault tolerance, power savings, etc.**
                                                   

In [ ]:
!pip install tf-agents[reverb]
!pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable
                                              0.0/1.4 MB ? eta -:--:--
     ---------------------------------------  1.4/1.4 MB 29.2 MB/s eta 0:00:01
     ---------------------------------------- 1.4/1.4 MB 17.5 MB/s eta 0:00:00
                                              0.0/277.3 kB ? eta -:--:--
     ------------------------------------- 277.3/277.3 kB 16.7 MB/s eta 0:00:00
                                              0.0/61.3 kB ? eta -:--:--
     ---------------------------------------- 61.3/61.3 kB ? eta 0:00:00
  Using cached gym-0.23.0-py3-none-any.whl
                                              0.0/10.4 MB ? eta -:--:--
     -------                                  1.9/10.4 MB 40.5 MB/s eta 0:00:01
     -------------                            3.5/10.4 MB 31.5 MB/s eta 0:00:01
     -------------------                      5.0/10.4 MB 31.8 MB/s eta 0:00:01
     ------------------------           

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [95 lines of output]
      
      
      WARNING, No "Setup" File Exists, Running "buildconfig/config.py"
      Using WINDOWS configuration...
      
      Making dir :prebuilt_downloads:
      Downloading... https://www.libsdl.org/release/SDL2-devel-2.0.16-VC.zip 13d952c333f3c2ebe9b7bc0075b4ad2f784e7584
      Unzipping :prebuilt_downloads\SDL2-devel-2.0.16-VC.zip:
      Downloading... https://www.libsdl.org/projects/SDL_image/release/SDL2_image-devel-2.0.5-VC.zip 137f86474691f4e12e76e07d58d5920c8d844d5b
      Unzipping :prebuilt_downloads\SDL2_image-devel-2.0.5-VC.zip:
      Downloading... https://www.libsdl.org/projects/SDL_ttf/release/SDL2_ttf-devel-2.0.15-VC.zip 1436df41ebc47ac36e02ec9bda5699e80ff9bd27
      Unzipping :prebuilt_downloads\SDL2_ttf-devel-2.0.15-VC.zip:
      Downloading... https://www.libsdl.org/projects/SDL_mixer/release/SDL2_mixer-devel-

Defaulting to user installation because normal site-packages is not writeable
                                              0.0/1.7 MB ? eta -:--:--
     ---------------------------              1.2/1.7 MB 15.1 MB/s eta 0:00:01
     ---------------------------------------- 1.7/1.7 MB 13.7 MB/s eta 0:00:00
                                              0.0/375.9 MB ? eta -:--:--
                                             2.1/375.9 MB 44.0 MB/s eta 0:00:09
                                             3.8/375.9 MB 30.6 MB/s eta 0:00:13
                                             5.1/375.9 MB 29.6 MB/s eta 0:00:13
                                             6.2/375.9 MB 28.3 MB/s eta 0:00:14
                                             7.8/375.9 MB 27.8 MB/s eta 0:00:14
     -                                       9.8/375.9 MB 27.2 MB/s eta 0:00:14
     -                                      11.4/375.9 MB 26.2 MB/s eta 0:00:14
     -                                      13.0/375.9 MB 26

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.0.247 requires numpy<2,>=1, but you have numpy 2.1.3 which is incompatible.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.1.3 which is incompatible.
streamlit 1.25.0 requires numpy<2,>=1.19.3, but you have numpy 2.1.3 which is incompatible.

[notice] A new release of pip is available: 22.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Importing necessary libraries**

In [3]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [5]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import abc
import tensorflow as tf
import numpy as np
import random

# TF‑Agents imports
from tf_agents.environments import py_environment
from tf_agents.environments import tf_environment
from tf_agents.environments import tf_py_environment
from tf_agents.environments import wrappers
from tf_agents.environments import suite_gym
from tf_agents.specs import array_spec
from tf_agents.trajectories import time_step as ts
from tf_agents.trajectories import trajectory
from tf_agents.networks import q_network
from tf_agents.agents.dqn import dqn_agent
from tf_agents.utils import common
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.drivers import dynamic_step_driver

ImportError: Traceback (most recent call last):
  File "C:\Users\Arshad\AppData\Roaming\Python\Python311\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

**REQUIREMENTS**

The agent will receive the observation from the different servers

1.   Latency in the response
2.   Number of requests handled by individual servers
3.   Number of failed requests (use a threshold value of maximum requests that can be handled by each server (it can be different for different servers depending on whether we're classifying it as level-1, level-2 or level-3 server))
4.   Resource usage of different servers

Thus, create an environment that takes these metrics as observation, and receive these values from different servers.


**Testing env**

In [ ]:
import requests

def getResponseFromServer():
    url = 'http://localhost:8005'

    try:
        response = requests.get(url)

        if response.status_code == 200:
            print("Successfully received response from load balancer")
            return
        else:
            print('Error:', response.status_code)
            return None
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
getResponseFromServer()

Success


In [ ]:
import requests

PROMETHEUS_URL = "http://your-prometheus-server:9090/api/v1/query"

def _generate_state(self):
    # Query Prometheus for real data
    query_latency = requests.get(f"{PROMETHEUS_URL}?query=avg_over_time(http_request_duration_seconds[5m])").json()
    query_requests = requests.get(f"{PROMETHEUS_URL}?query=sum(rate(http_requests_total[5m]))").json()

    # Parse the response (assuming 3 servers)
    latency = [float(result["value"][1]) for result in query_latency["data"]["result"][:self._num_servers]]
    requests_handled = [float(result["value"][1]) for result in query_requests["data"]["result"][:self._num_servers]]

    # Simulate failed requests & resource usage for now
    failed_requests = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))
    resource_usage = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))

    state = np.column_stack([latency, requests_handled, failed_requests, resource_usage])
    return state.astype(np.float32)

In [ ]:
class LoadBalancerEnv(py_environment.PyEnvironment):
    def __init__(self, num_servers=3):
        super(LoadBalancerEnv, self).__init__()
        self._num_servers = num_servers

        # Define thresholds (maximum requests that can be handled) per server.
        # These can differ based on server capacity (light, medium, very good).
        self._max_requests = np.array([100, 200, 300])[:num_servers]

        # Observation: For each server, we have 4 metrics.
        # We represent them as a (num_servers, 4) float32 array.
        self._observation_spec = array_spec.BoundedArraySpec(
            shape=(self._num_servers, 4),
            dtype=np.float32,
            minimum=0.0,
            maximum=1.0,
            name='observation'
        )

        # Action: The agent outputs a vector of weights for each server.
        # We define a continuous action space where each element is in [0, 1].
        # (The agent is responsible for normalizing these values, or you can normalize them in step.)
        self._action_spec = array_spec.BoundedArraySpec(
            shape=(self._num_servers,),
            dtype=np.float32,
            minimum=0.0,
            maximum=1.0,
            name='action'
        )

        self._episode_ended = False
        self._step_count = 0
        self._max_steps = 50  # Define the length of an episode

        # Initialize the state (randomized for simulation)
        self._state = self._generate_state()

    def _generate_state(self):
        """
        Simulate state generation.
        Each server metric is normalized between 0 and 1.
        For instance:
          - Latency: 0 (fast) to 1 (slow)
          - Requests handled: 0 (none) to 1 (at capacity)
          - Failed requests: 0 (none) to 1 (maximum failures)
          - Resource usage: 0 (idle) to 1 (fully used)
        """
        # Here we use random values for demonstration.
        # In a real system, you’d sample these from actual server metrics.
        latency = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))
        requests_handled = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))
        failed_requests = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))
        resource_usage = np.random.uniform(0.0, 1.0, size=(self._num_servers, 1))
        state = np.concatenate([latency, requests_handled, failed_requests, resource_usage], axis=1)
        return state.astype(np.float32)

    def action_spec(self):
        return self._action_spec

    def observation_spec(self):
        return self._observation_spec

    def _reset(self):
        self._episode_ended = False
        self._step_count = 0
        self._state = self._generate_state()
        return ts.restart(self._state)

    def _step(self, action):
        if self._episode_ended:
            return self.reset()

        self._step_count += 1

        # Normalize the action to get a proper distribution of traffic.
        if np.sum(action) > 0:
            traffic_distribution = action / np.sum(action)
        else:
            # Avoid division by zero by assigning equal weights.
            traffic_distribution = np.ones(self._num_servers) / self._num_servers

        # --- Simulate Environment Dynamics ---
        #
        # For each server, assume that higher allocated traffic increases load.
        # We'll simulate the impact by updating the state metrics:
        #
        # 1. Latency increases with traffic (simulate congestion).
        # 2. Requests handled increases based on allocated traffic.
        # 3. Failed requests increase if requests handled are near the threshold.
        # 4. Resource usage increases with traffic allocation.
        #
        # For simplicity, we use basic updates:
        delta = traffic_distribution.reshape(-1, 1) * 0.1  # scaling factor

        # Update each metric slightly.
        # In a realistic scenario, you would use more sophisticated dynamics.
        self._state[:, 0] = np.clip(self._state[:, 0] + delta[:, 0] + np.random.normal(0, 0.01, size=self._num_servers), 0, 1)  # latency
        self._state[:, 1] = np.clip(self._state[:, 1] + delta[:, 0] + np.random.normal(0, 0.01, size=self._num_servers), 0, 1)  # requests handled
        self._state[:, 2] = np.clip(self._state[:, 2] + (delta[:, 0] if np.any(self._state[:, 1] > 0.9) else 0) + np.random.normal(0, 0.01, size=self._num_servers), 0, 1)  # failed requests
        self._state[:, 3] = np.clip(self._state[:, 3] + delta[:, 0] + np.random.normal(0, 0.01, size=self._num_servers), 0, 1)  # resource usage

        # --- Compute the Reward ---
        #
        # A simple reward function:
        # - Penalize high latency and high resource usage.
        # - Penalize when the number of requests handled gets close to or exceeds the server's threshold.
        #
        # We simulate the "overload" condition by comparing the normalized requests_handled to 0.9.
        latency_penalty = np.mean(self._state[:, 0])
        resource_penalty = np.mean(self._state[:, 3])
        overload_penalty = np.mean(self._state[:, 1] > 0.9).astype(np.float32)

        reward = - (latency_penalty + resource_penalty + overload_penalty)

        # End episode if we have run the maximum steps.
        if self._step_count >= self._max_steps:
            self._episode_ended = True
            return ts.termination(self._state, reward)
        else:
            return ts.transition(self._state, reward=reward, discount=0.99)


In [ ]:
GAMMA = 0.99
NEURAL_NETWORK_HIDDEN_UNITS = (128, 64, 32)
MAX_BUFFER_SIZE = 30000
LEARNING_RATE = 1e-3
MAX_STEPS_ALLOWED_IN_GRID = 1_00_000

In [ ]:
# -----------------------------------------------------------------------------
# 2. Create the TF-Agents Environment and the DQN Agent
# -----------------------------------------------------------------------------
# Create a Python environment and wrap it as a TF-PyEnvironment.
# create an instance of the CardGameEnv class

grid_env = LoadBalancerEnv()
train_grid_env = tf_py_environment.TFPyEnvironment(grid_env)

# Create a QNetwork. Here we use a simple two-layer fully connected network.
fc_layer_params = NEURAL_NETWORK_HIDDEN_UNITS
q_net = q_network.QNetwork(
    train_grid_env.observation_spec(),
    train_grid_env.action_spec(),
    fc_layer_params=fc_layer_params)

# Use an optimizer. (TF-Agents examples often use tf.compat.v1.train.AdamOptimizer.)
optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate=LEARNING_RATE)

# Create a counter to keep track of the training steps.
train_step_counter = tf.Variable(0)

# Instantiate the DQN Agent.
# agent is passed with parameters -
# time_step_spec, action_spec, q_network,
# opimizer, error_function, train_Step_Counter

agent = dqn_agent.DqnAgent(
    grid_env.time_step_spec(),
    grid_env.action_spec(),
    q_network=q_net,
    optimizer=optimizer,
    td_errors_loss_fn=common.element_wise_squared_loss,
    train_step_counter=train_step_counter)

agent.initialize()

In [ ]:
# ---------------------------------------------------------------------------
# 3. Setup the Replay Buffer and Data Collection for Treasure Hunt
# ---------------------------------------------------------------------------

# Create a replay buffer to store experience for Treasure Hunt
replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=agent.collect_data_spec,
    batch_size=train_grid_env.batch_size,
    max_length=MAX_BUFFER_SIZE
)

# Function to collect a single step of experience in Treasure Hunt
def collect_step(environment, policy, buffer):
    time_step = environment.current_time_step()  # Get current state
    action_step = policy.action(time_step)  # Agent selects an action
    next_time_step = environment.step(action_step.action)  # Perform action
    traj = trajectory.from_transition(time_step, action_step, next_time_step)  # Store transition

    # Add transition to the replay buffer
    buffer.add_batch(traj)

# Collect some initial experience for the agent to learn from
initial_collect_steps = 1500  # Increase to ensure proper training

for _ in range(initial_collect_steps):
    collect_step(train_grid_env, agent.collect_policy, replay_buffer)

# Create a dataset from the replay buffer for training
dataset = replay_buffer.as_dataset(
    sample_batch_size=20,  # Adjust batch size for stability
    num_steps=2  # Using 2 time steps to improve learning
).prefetch(3)  # Prefetch for performance

# Create an iterator for sampling batches
iterator = iter(dataset)


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

# -----------------------------------------------------------------------------
# 4. Training the Agent with Visualization
# -----------------------------------------------------------------------------

num_iterations = 600
train_loss_history = []

plt.ion()  # Turn on interactive mode for live updates

for step in range(num_iterations):
    # Collect one step and add it to the replay buffer.
    collect_step(train_grid_env, agent.collect_policy, replay_buffer)

    # Sample a batch of experiences from the replay buffer.
    experience, unused_info = next(iterator)
    train_loss = agent.train(experience).loss.numpy()

    train_loss_history.append(train_loss)

    train_step_counter.assign_add(1)

    if train_step_counter.numpy() % 100 == 0:
        print(f'Step = {train_step_counter.numpy()}: Loss = {train_loss}')

        clear_output(wait=True)  # Clears previous output for smooth animation
        plt.figure(figsize=(10, 5))
        plt.plot(train_loss_history, label="Training Loss", color="blue")
        plt.xlabel("Training Steps")
        plt.ylabel("Loss")
        plt.title("Training Loss Over Time")
        plt.legend()
        plt.grid(True)
        plt.show()

plt.ioff()

In [ ]:
# -----------------------------------------------------------------------------
# 5. Evaluate the Trained Agent and Plot Returns
# -----------------------------------------------------------------------------
num_eval_episodes = 1
eval_returns = []  # To store returns for each evaluation episode

for episode in range(num_eval_episodes):
    time_step = train_grid_env.reset()
    episode_return = 0.0

    while not time_step.is_last():
        # Use the agent's (greedy) policy for evaluation.
        action_step = agent.policy.action(time_step)
        time_step = train_grid_env.step(action_step.action)

        updated_grid = train_grid_env.pyenv.envs[0]._grid
        agent_position = train_grid_env.pyenv.envs[0]._x, train_grid_env.pyenv.envs[0]._y

        # time_step.reward is a tf.Tensor with shape [batch_size] (here, batch_size == 1)
        episode_return += time_step.reward.numpy()[0]

    eval_returns.append(episode_return)
    print('Evaluation Episode {}: Return = {}'.format(episode + 1, episode_return))

# Plot the evaluation returns
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_eval_episodes+1), eval_returns, marker='o', linestyle='-')
plt.title('Evaluation Returns per Episode')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.grid(True)
plt.show()

*Saving the Reinforcement Learning agent policy*

In [ ]:
from tf_agents.policies import policy_saver

# Specify the directory where you want to save the policy.
# You can use an absolute or relative path.
policy_dir = os.path.join("saved_policies", "load_balancing_trained_policy")
os.makedirs(policy_dir, exist_ok=True)

# Create a PolicySaver using the agent's policy.
tf_policy_saver = policy_saver.PolicySaver(agent.policy)

# Save the policy to the directory.
tf_policy_saver.save(policy_dir)
print(f"Policy saved at: {policy_dir}")


# Using the RL agent policy

In [ ]:
# import tensorflow as tf
# from tf_agents.policies import py_tf_policy

# # Load the saved policy from disk.
# policy_dir = "saved_policies/treasure_hunt_policy"
# loaded_policy = tf.compat.v2.saved_model.load(policy_dir)

# # Now use the wrapped policy in your application:
# time_step = train_grid_env.reset()  # your TFPyEnvironment instance
# while not time_step.is_last():
#     action_step = loaded_policy.action(time_step)
#     time_step = train_grid_env.step(action_step.action)
#     print("Action taken:", action_step.action.numpy())
